In [ ]:
# Cell 1: Import thư viện
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Cell 2: Cấu hình và Tải dữ liệu
CLEANED_DATA_PATH = "../data/processed/cleaned_uk_data.csv"
RULES_PATH = "../data/processed/rules_fpgrowth_filtered.csv"

df_clean = pd.read_csv(CLEANED_DATA_PATH)
df_rules = pd.read_csv(RULES_PATH)

# Tính giá trị trung bình mỗi mặt hàng và tổng số hóa đơn
item_prices = df_clean.groupby('Description')['UnitPrice'].mean().to_dict()
total_invoices = df_clean['InvoiceNo'].nunique()

# Cell 3: Tính toán trọng số kinh tế (Khác biệt hoàn toàn ví dụ)
def compute_business_metrics(row):
    # Parse items từ chuỗi (hỗ trợ cả định dạng frozenset)
    def parse(txt):
        return [i.strip(" '{}") for i in txt.replace("frozenset(", "").replace(")", "").split(",")]
    
    all_items = parse(row['antecedents']) + parse(row['consequents'])
    
    # 1. Giá trị trung bình của gói hàng (Weight)
    basket_val = sum([item_prices.get(it, 0) for it in all_items])
    
    # 2. Doanh thu ước tính (Real Impact) = (Support * Total Bills) * Basket Value
    est_revenue = (row['support'] * total_invoices) * basket_val
    
    return pd.Series([basket_val, est_revenue])

df_rules[['basket_value', 'est_revenue']] = df_rules.apply(compute_business_metrics, axis=1)

# Cell 4: Phân khúc 3 nhóm theo phân vị (Quantile Segmentation)
# Không dùng ngưỡng cứng, dùng Top 25% để tìm luật VIP
val_high = df_rules['basket_value'].quantile(0.75)
lift_high = df_rules['lift'].quantile(0.75)

def classify(row):
    if row['lift'] >= lift_high and row['basket_value'] >= val_high:
        return 'Nhóm 1: VIP Rules (Lợi nhuận & Gắn kết cao)'
    elif row['basket_value'] >= val_high:
        return 'Nhóm 2: Niche/Luxury (Giá trị cao)'
    else:
        return 'Nhóm 3: Mass Market (Hàng phổ thông)'

df_rules['segment'] = df_rules.apply(classify, axis=1)

# Cell 5: Trực quan hóa Ma trận Chiến lược

plt.figure(figsize=(12, 7))
sns.scatterplot(data=df_rules, x='lift', y='basket_value', 
                hue='segment', size='est_revenue', sizes=(50, 600), alpha=0.6)
plt.axvline(lift_high, color='grey', linestyle='--')
plt.axhline(val_high, color='grey', linestyle='--')
plt.title("Phân tích Ma trận Luật kết hợp: Lift vs. Giá trị đơn hàng")
plt.xlabel("Sức mạnh liên kết (Lift)")
plt.ylabel("Giá trị trung bình gói hàng (£)")
plt.legend(title="Phân khúc")
plt.show()